# Figure aggiuntive — U-Net SWE Downscaling

Questo notebook produce le 4 figure aggiuntive per il paper:
- **Fig. 7** — Scatter SWE predetto vs osservato per bacino (40 anni OOS)
- **Fig. 8** — Mappa spaziale side-by-side SPORTLIS vs U-Net (1 aprile anno scelto)
- **Fig. 9** — Curve di loss training/validation dai log PBS
- **Fig. 10** — Bias (pred − obs) vs quantili di SWE osservato

Eseguire su NCAR JupyterHub con kernel `sportlis-unet`.

In [ ]:
# ── Cell 1: Import e path ──────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import xarray as xr
import torch
import torch.nn as nn
import torch.nn.functional as F
import re
from pathlib import Path
from scipy.ndimage import uniform_filter
from scipy import stats

# ── Paths NCAR ────────────────────────────────────────────────────────────
OUTPUT_DIR  = Path('/glade/u/home/cionni/unet_sportlis/output_extended')
LOG_DIR     = Path('/glade/u/home/cionni/unet_sportlis/logs')
MEMMAP_DIR  = Path('/glade/work/cionni/sportlis_data/memmap_extended')
STATIC_FILE = Path('/glade/work/cionni/sportlis_data/sportlis_static_extended.nc')
CANOPY_FILE = Path('/glade/work/cionni/sportlis_data/sportlis_canopy_extended_3km.nc')
CKPT_TPL    = 'best_unet_extended_LOYO_test{year}.pt'
BASIN_CSV   = OUTPUT_DIR / 'basin_ts_results.csv'
FIG_DIR     = OUTPUT_DIR / 'figures_additional'
FIG_DIR.mkdir(exist_ok=True)

# ── Palette bacini ────────────────────────────────────────────────────────
BASINS = {
    'Pacific_NW_Cascades':  ('Pacific NW Cascades',  '#1f77b4'),
    'Oregon_Cascades':      ('Oregon Cascades',       '#17becf'),
    'Sierra_Nevada':        ('Sierra Nevada',          '#ff7f0e'),
    'Northern_Rockies':     ('Northern Rockies',       '#2ca02c'),
    'Southern_Rockies':     ('Southern Rockies (CO)',  '#d62728'),
    'Idaho':                ('Idaho (dom. originale)', '#9467bd'),
}

ALL_YEARS = list(range(1985, 2025))
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Basin CSV: {BASIN_CSV.exists()}')

In [ ]:
# ── Cell 2: Carica basin_ts_results.csv ───────────────────────────────────
df = pd.read_csv(BASIN_CSV)
# Normalizza nome colonne per compatibilità (vecchio/nuovo schema)
df.columns = df.columns.str.strip()
if 'fold_used' not in df.columns:
    df['fold_used'] = df['year']
    df['is_oos']    = True
df['is_oos'] = df['is_oos'].astype(str).str.lower().isin(['true', '1', 'yes'])
df_oos = df[df['is_oos']].copy()
print(f'Totale righe: {len(df)}  |  OOS: {len(df_oos)}')
print(f'Anni OOS: {sorted(df_oos.year.unique())}')
df_oos.head(3)

In [ ]:
# ── Cell 3: FIG. 7 — Scatter pred vs obs per bacino ───────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

metrics_scatter = {}
for ax, (basin_key, (basin_label, color)) in zip(axes, BASINS.items()):
    sub = df_oos[df_oos['basin'] == basin_key].dropna(subset=['pred_mean', 'obs_mean'])
    if sub.empty:
        ax.set_visible(False)
        continue

    x = sub['obs_mean'].values
    y = sub['pred_mean'].values

    # Scatter colorato per anno
    sc = ax.scatter(x, y, c=sub['year'], cmap='RdYlGn_r', s=60,
                    edgecolors='k', linewidths=0.4, zorder=3)
    plt.colorbar(sc, ax=ax, label='Anno', shrink=0.85)

    # Linea 1:1
    lim_max = max(x.max(), y.max()) * 1.1
    ax.plot([0, lim_max], [0, lim_max], 'k--', lw=1.2, label='1:1', zorder=2)

    # Regressione lineare
    if len(x) > 2:
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xfit = np.linspace(0, lim_max, 100)
        ax.plot(xfit, slope * xfit + intercept, color=color, lw=2,
                label=f'fit: y={slope:.2f}x+{intercept:.2f}\nr={r:.2f}', zorder=4)
        metrics_scatter[basin_key] = dict(slope=slope, intercept=intercept, r=r)

    ax.set_xlabel('SWE osservato (mm)', fontsize=10)
    ax.set_ylabel('SWE predetto (mm)', fontsize=10)
    ax.set_title(basin_label, color=color, fontweight='bold', fontsize=11)
    ax.legend(fontsize=8, loc='upper left')
    ax.set_xlim(0, lim_max)
    ax.set_ylim(0, lim_max)
    ax.grid(True, alpha=0.3)

fig.suptitle('Fig. 7 — Scatter SWE predetto vs osservato (OOS LOYO-40)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
out7 = FIG_DIR / 'fig7_scatter_pred_obs.png'
fig.savefig(out7, dpi=150, bbox_inches='tight')
plt.show()
print(f'Salvato: {out7}')

In [ ]:
# ── Cell 4: FIG. 10 — Bias vs quantili SWE osservato ─────────────────────
# Usa tutte le osservazioni OOS aggregate per bacino
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

# Per una versione pixel-level servirebbero le predizioni grezze;
# qui usiamo i valori aggregati per anno (media bacino) dai dati basin_ts.
# Per una versione pixel-level, decommenta Cell 5 (richiede GPU).

for ax, (basin_key, (basin_label, color)) in zip(axes, BASINS.items()):
    sub = df_oos[df_oos['basin'] == basin_key].dropna(subset=['pred_mean', 'obs_mean'])
    if sub.empty:
        ax.set_visible(False)
        continue

    obs  = sub['obs_mean'].values
    bias = sub['pred_mean'].values - obs  # positivo = sovrastima

    # Ordina per obs
    idx  = np.argsort(obs)
    obs_sorted  = obs[idx]
    bias_sorted = bias[idx]

    ax.scatter(obs_sorted, bias_sorted, c=sub['year'].values[idx],
               cmap='RdYlGn_r', s=60, edgecolors='k', linewidths=0.4, zorder=3)
    ax.axhline(0, color='k', lw=1.2, ls='--', zorder=2)

    # Moving average del bias
    if len(obs_sorted) >= 5:
        win = max(3, len(obs_sorted) // 5)
        bias_smooth = np.convolve(bias_sorted, np.ones(win)/win, mode='valid')
        obs_smooth  = np.convolve(obs_sorted,  np.ones(win)/win, mode='valid')
        ax.plot(obs_smooth, bias_smooth, color=color, lw=2.5, label=f'media mobile (w={win})')

    ax.set_xlabel('SWE osservato (mm)', fontsize=10)
    ax.set_ylabel('Bias pred−obs (mm)', fontsize=10)
    ax.set_title(basin_label, color=color, fontweight='bold', fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Fig. 10 — Bias (pred−obs) vs SWE osservato — anni OOS LOYO-40',
             fontsize=13, fontweight='bold')
plt.tight_layout()
out10 = FIG_DIR / 'fig10_bias_vs_swe.png'
fig.savefig(out10, dpi=150, bbox_inches='tight')
plt.show()
print(f'Salvato: {out10}')

In [ ]:
# ── Cell 5: FIG. 9 — Curve di loss dai log PBS ───────────────────────────
# Sceglie alcuni fold rappresentativi da plottare
FOLD_SAMPLE = [0, 5, 10, 20, 30, 39]   # indici fold (corrispondono agli anni 1985,1990,...)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

pat_epoch = re.compile(
    r'E(\d+)/\d+\s+train=([\d.]+)\s+val=([\d.]+)')

for ax, fi in zip(axes, FOLD_SAMPLE):
    log = LOG_DIR / f'loyo40_job{fi}.log'
    if not log.exists():
        ax.set_title(f'fold {fi} — log mancante', color='gray')
        ax.set_visible(False)
        continue

    text = log.read_text(errors='ignore')
    epochs, train_loss, val_loss = [], [], []
    for m in pat_epoch.finditer(text):
        epochs.append(int(m.group(1)))
        train_loss.append(float(m.group(2)))
        val_loss.append(float(m.group(3)))

    if not epochs:
        ax.set_title(f'fold {fi} (anno {1985+fi}) — nessuna epoca trovata', color='gray')
        continue

    # Anno test
    test_year_match = re.search(r'test=(\d{4})', text)
    test_yr = test_year_match.group(1) if test_year_match else str(1985 + fi)

    ax.plot(epochs, train_loss, 'o-', color='steelblue', ms=3, lw=1.5, label='Train loss')
    ax.plot(epochs, val_loss,   's--', color='tomato',   ms=3, lw=1.5, label='Val loss')
    # Best epoch
    best_ep = epochs[np.argmin(val_loss)]
    ax.axvline(best_ep, color='k', lw=1, ls=':', alpha=0.7)
    ax.text(best_ep + 0.3, ax.get_ylim()[0] * 1.05 if ax.get_ylim()[0] > 0 else 0.05,
            f'best\nep={best_ep}', fontsize=7, va='bottom')

    ax.set_xlabel('Epoca', fontsize=9)
    ax.set_ylabel('Loss (Huber, log1p)', fontsize=9)
    ax.set_title(f'Fold test={test_yr}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Fig. 9 — Curve di convergenza (loss train/val) — fold selezionati',
             fontsize=13, fontweight='bold')
plt.tight_layout()
out9 = FIG_DIR / 'fig9_loss_curves.png'
fig.savefig(out9, dpi=150, bbox_inches='tight')
plt.show()
print(f'Salvato: {out9}')

In [ ]:
# ── Cell 6: Architettura U-Net (copia esatta da train_loyo40.py) ──────────
# Necessaria per Fig. 8 (mappa spaziale)

def pad_to_mult(x, mult=8):
    _, _, H, W = x.shape
    pH = (mult - H % mult) % mult
    pW = (mult - W % mult) % mult
    x = F.pad(x, (0, pW, 0, pH))
    return x, (pH, pW)

def unpad(x, pads):
    pH, pW = pads
    H, W = x.shape[-2], x.shape[-1]
    return x[..., :H - pH if pH else H, :W - pW if pW else W]

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        g = min(8, out_ch)
        while out_ch % g != 0:
            g -= 1
        layers = [
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.GroupNorm(g, out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.GroupNorm(g, out_ch), nn.GELU()
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels, out_channels=1, base=32, dropout=0.1):
        super().__init__()
        self.e1 = DoubleConv(in_channels, base)
        self.p1 = nn.MaxPool2d(2)
        self.e2 = DoubleConv(base, base*2)
        self.p2 = nn.MaxPool2d(2)
        self.e3 = DoubleConv(base*2, base*4)
        self.p3 = nn.MaxPool2d(2)
        self.bn = DoubleConv(base*4, base*8, dropout=dropout)
        self.u1 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2)
        self.d1 = DoubleConv(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.d2 = DoubleConv(base*4, base*2)
        self.u3 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.d3 = DoubleConv(base*2, base)
        self.out = nn.Conv2d(base, out_channels, 1)

    def forward(self, x):
        xp, pads = pad_to_mult(x)
        e1 = self.e1(xp)
        e2 = self.e2(self.p1(e1))
        e3 = self.e3(self.p2(e2))
        b  = self.bn(self.p3(e3))
        d  = self.d1(torch.cat([self.u1(b),  e3], 1))
        d  = self.d2(torch.cat([self.u2(d),  e2], 1))
        d  = self.d3(torch.cat([self.u3(d),  e1], 1))
        return F.softplus(unpad(self.out(d), pads))

N_FEAT = 16  # canali input totali
print('Architettura U-Net caricata.')

In [ ]:
# ── Cell 7: Funzione load_static (uguale a train_loyo40.py) ───────────────
def load_static():
    ds = xr.open_dataset(STATIC_FILE)
    elev_raw = ds['elevation'].values.astype(np.float32)
    slope    = ds['slope'].values.astype(np.float32)
    asp_s    = ds['aspect_sin'].values.astype(np.float32)
    asp_c    = ds['aspect_cos'].values.astype(np.float32)
    lat      = ds['lat'].values.astype(np.float32) if 'lat' in ds else None
    lon      = ds['lon'].values.astype(np.float32) if 'lon' in ds else None
    ds.close()

    ds_c   = xr.open_dataset(CANOPY_FILE)
    canopy = ds_c['canopy_fraction'].values.astype(np.float32)
    ds_c.close()

    elev_filled = np.where(np.isfinite(elev_raw), elev_raw, np.nanmedian(elev_raw))
    tpi_s = (elev_filled - uniform_filter(elev_filled, size=11, mode='nearest')).astype(np.float32)
    tpi_l = (elev_filled - uniform_filter(elev_filled, size=21, mode='nearest')).astype(np.float32)

    def norm(arr):
        m, s = float(np.nanmean(arr)), float(np.nanstd(arr))
        return ((arr - m) / (s if s > 0 else 1.0)).astype(np.float32)

    elev  = np.nan_to_num(norm(elev_raw), nan=0.0)
    slope = np.nan_to_num(norm(slope),    nan=0.0)
    tpi_s = np.nan_to_num(norm(tpi_s),    nan=0.0)
    tpi_l = np.nan_to_num(norm(tpi_l),    nan=0.0)
    asp_s = np.where(np.isfinite(asp_s), asp_s, 0.0).astype(np.float32)
    asp_c = np.where(np.isfinite(asp_c), asp_c, 0.0).astype(np.float32)
    canopy = np.nan_to_num(
        (canopy - float(np.nanmean(canopy))).astype(np.float32), nan=0.0)

    topo = np.stack([elev, slope, asp_s, asp_c, tpi_s, tpi_l, canopy], axis=0)  # (7,H,W)

    # Lat/lon normalizzate
    H, W = elev.shape
    if lat is None:
        lat_g = np.linspace(0, 1, H, dtype=np.float32)
        lon_g = np.linspace(0, 1, W, dtype=np.float32)
        lat_n = np.tile(lat_g[:, None], (1, W))
        lon_n = np.tile(lon_g[None, :], (H, 1))
    else:
        lat_n = (lat - lat.mean()) / (lat.std() + 1e-6)
        lon_n = (lon - lon.mean()) / (lon.std() + 1e-6)

    return topo, lat_n, lon_n

print('Carico features statiche...')
topo_static, lat_norm, lon_norm = load_static()
print(f'topo shape: {topo_static.shape}  lat: {lat_norm.shape}')

In [ ]:
# ── Cell 8: Funzione carica norm stats e modello ──────────────────────────
def load_norm_stats(test_year):
    """Carica mean/std dal cache npz oppure dai log."""
    cache_file = OUTPUT_DIR / f'norm_stats_cache_test{test_year}.npz'
    if cache_file.exists():
        d = np.load(str(cache_file))
        return d['mean_arr'], d['std_arr']
    # Fallback: legge dal log
    fi = test_year - 1985
    log = LOG_DIR / f'loyo40_job{fi}.log'
    if not log.exists():
        raise FileNotFoundError(f'Né cache né log trovati per test_year={test_year}')
    text = log.read_text(errors='ignore')
    m = re.search(r'Stats: mean=\[([^\]]+)\]\s+std=\[([^\]]+)\]', text)
    if not m:
        raise ValueError(f'Stats non trovate nel log {log}')
    mean_arr = np.array([float(v) for v in m.group(1).split()], dtype=np.float32)
    std_arr  = np.array([float(v) for v in m.group(2).split()], dtype=np.float32)
    return mean_arr, std_arr

def load_model(test_year):
    ckpt_path = OUTPUT_DIR / CKPT_TPL.format(year=test_year)
    model = UNet(in_channels=N_FEAT).to(device)
    model.load_state_dict(torch.load(str(ckpt_path), map_location=device))
    model.eval()
    return model

print('Funzioni load_norm_stats e load_model definite.')

In [ ]:
# ── Cell 9: FIG. 8 — Mappa spaziale SWE side-by-side ─────────────────────
# Scegli un anno OOS e una data (indice temporale) da visualizzare.
# Consigliati: 1993 (anno molto nevoso), 2015 (anno secco), 1995 (record)

MAP_TEST_YEAR = 1993      # anno test (OOS): il modello non ha visto quest'anno
MAP_TARGET_DOY = 91       # giorno dell'anno (91 = 1 aprile circa)

# Carica memmap dell'anno selezionato
feat_file = MEMMAP_DIR / f'y{MAP_TEST_YEAR}_feat.npy'
mask_file = MEMMAP_DIR / f'y{MAP_TEST_YEAR}_mask.npy'
days_file = MEMMAP_DIR / f'y{MAP_TEST_YEAR}_days.npy'

feat_mm = np.load(str(feat_file), mmap_mode='r')   # (T, 8, H, W)
mask_mm = np.load(str(mask_file), mmap_mode='r')   # (T, H, W)  — 1=snow pixel
days_mm = np.load(str(days_file), allow_pickle=True)

T, C, H, W = feat_mm.shape
print(f'Anno {MAP_TEST_YEAR}: shape={feat_mm.shape}  giorni={T}')

# Trova il timestamp più vicino al DOY target
doys = pd.DatetimeIndex(days_mm).day_of_year
t_idx = int(np.argmin(np.abs(doys - MAP_TARGET_DOY)))
target_date = pd.Timestamp(days_mm[t_idx])
print(f'Timestamp scelto: {target_date.date()}  (DOY={doys[t_idx]})')

In [ ]:
# ── Cell 10: Costruisce input tensor e fa forward pass ────────────────────
mean_arr, std_arr = load_norm_stats(MAP_TEST_YEAR)
# mean/std sono per i 7 canali temporali (ch 0-6 del memmap)
N_TM = 7

feat_t = feat_mm[t_idx].copy()   # (8, H, W)
# Canali 0-6: normalizza. Canale 7: SWE target (non usato come input)
feat_norm = feat_t[:N_TM].copy()
for c in range(N_TM):
    feat_norm[c] = np.nan_to_num(
        (feat_norm[c] - mean_arr[c]) / (std_arr[c] if std_arr[c] > 0 else 1.0),
        nan=0.0, posinf=0.0, neginf=0.0)

# Assembla 16 canali: 7 TM + lat + lon + 7 topo
lat_ch  = lat_norm[np.newaxis]   # (1,H,W)
lon_ch  = lon_norm[np.newaxis]
inp = np.concatenate([feat_norm, lat_ch, lon_ch, topo_static], axis=0)  # (16,H,W)

x_t = torch.from_numpy(inp[np.newaxis]).to(device)   # (1,16,H,W)

model = load_model(MAP_TEST_YEAR)
with torch.no_grad():
    pred_log = model(x_t).cpu().numpy()[0, 0]   # (H,W) in log1p space
pred_mm = np.expm1(pred_log)   # → mm

# SWE osservato (raw mm)
obs_raw = feat_mm[t_idx, 7]    # ch 7 = swe_target_filled
domain_mask = mask_mm[t_idx]   # 1 = pixel nel dominio

print(f'Pred  → min={pred_mm.min():.2f}  max={pred_mm.max():.2f}  mean={pred_mm[domain_mask>0].mean():.2f} mm')
print(f'Obs   → min={obs_raw.min():.2f}  max={obs_raw.max():.2f}  mean={obs_raw[domain_mask>0].mean():.2f} mm')

In [ ]:
# ── Cell 11: Plot mappa spaziale ──────────────────────────────────────────
# Clip SWE per visualizzazione (rimuove outlier estremi)
vmax = np.nanpercentile(obs_raw[domain_mask > 0], 98)
cmap = plt.cm.Blues

# Maschera dominio
pred_plot = np.where(domain_mask > 0, pred_mm, np.nan)
obs_plot  = np.where(domain_mask > 0, obs_raw,  np.nan)
diff_plot = np.where(domain_mask > 0, pred_mm - obs_raw, np.nan)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Osservato
im0 = axes[0].imshow(obs_plot, origin='upper', cmap=cmap, vmin=0, vmax=vmax)
plt.colorbar(im0, ax=axes[0], label='SWE (mm)', shrink=0.85)
axes[0].set_title(f'SPORTLIS osservato\n{target_date.date()}', fontsize=11, fontweight='bold')
axes[0].axis('off')

# Predetto
im1 = axes[1].imshow(pred_plot, origin='upper', cmap=cmap, vmin=0, vmax=vmax)
plt.colorbar(im1, ax=axes[1], label='SWE (mm)', shrink=0.85)
axes[1].set_title(f'U-Net predetto (OOS test={MAP_TEST_YEAR})\n{target_date.date()}',
                   fontsize=11, fontweight='bold')
axes[1].axis('off')

# Differenza
dmax = np.nanpercentile(np.abs(diff_plot[np.isfinite(diff_plot)]), 95)
im2 = axes[2].imshow(diff_plot, origin='upper', cmap='RdBu_r', vmin=-dmax, vmax=dmax)
plt.colorbar(im2, ax=axes[2], label='Bias pred−obs (mm)', shrink=0.85)
axes[2].set_title('Differenza (pred − obs)', fontsize=11, fontweight='bold')
axes[2].axis('off')

fig.suptitle(f'Fig. 8 — Mappa SWE spaziale — {target_date.strftime("%d %b %Y")} — anno test OOS={MAP_TEST_YEAR}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
out8 = FIG_DIR / f'fig8_map_spatial_{MAP_TEST_YEAR}_{target_date.strftime("%m%d")}.png'
fig.savefig(out8, dpi=150, bbox_inches='tight')
plt.show()
print(f'Salvato: {out8}')

In [ ]:
# ── Cell 12: Riepilogo figure prodotte ────────────────────────────────────
import os
print('=== Figure prodotte ===')
for f in sorted(FIG_DIR.glob('*.png')):
    size_kb = f.stat().st_size // 1024
    print(f'  {f.name:50s}  {size_kb:4d} KB')

print(f'\nCartella output: {FIG_DIR}')